### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="lending_club_1m",
    version_from_unique_name="lending_club",
    version_comment="""
To sub-sample this dataset, we only take the first year to obtain 814751 train and 276431.
""",
    # same as lending_club.ipynb
    dataset_year="2018",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://zenodo.org/records/11295916",
    download_description="""
It seems the official download portal from https://www.lendingclub.com is gone. So we can only utilize artifacts from the past, which seem to go up to 2018.
Old script and source: https://github.com/nateGeorge/preprocess_lending_club_data

From all the artifacts we found online, this seems to be the newest: https://www.kaggle.com/datasets/wordsforthewise/lending-club/data
And the following paper (https://arxiv.org/abs/2401.16458) curated this dataset https://zenodo.org/records/11295916 for tabular-text learning.

Other artifacts:
- https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data-csv
- https://www.kaggle.com/datasets/imsparsh/lending-club-loan-dataset-2007-2011

wget https://zenodo.org/records/11295916/files/LC_loans_granting_model_dataset.csv?download=1
mkdir -p local-data-warehouse/lending_club && mv LC_loans_granting_model_dataset.csv?download=1 local-data-warehouse/lending_club
kaggle datasets download wordsforthewise/lending-club -f accepted_2007_to_2018Q4.csv.gz && mv accepted_2007_to_2018Q4.csv.gz local-data-warehouse/lending_club/
""",
    # References
    academic_reference_bibtex=r"""@article{sanz2025credit,
  title={Credit Risk Meets Large Language Models: Building a Risk Indicator from Loan Descriptions in P2P Lending},
  author={Sanz-Guerrero, Mario and Arroyo, Javier},
  journal={Inteligencia Artificial},
  volume={28},
  number={75},
  pages={220--247},
  year={2025}
}
""",
    academic_reference_bibtex_key="sanz2025credit",
    license="CC0: Public Domain", # On Kaggle, likely not true TOS from the website.
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
We start withe data from Sanz-Guerrero et al. (2025). This data is already well preprocessed and curated for the task and used for tabular-text learning. However, we noticed a lot variables from the original data are missing that can be added to the task without invalidating the task or introducing data leakage. Thus, we also merge new features from the original data into the version from Sanz-Guerrero et al. (2025).

- This is a datasets where the description is very often empty or a standard phrase. So the model needs to be able to handle cases with a lot of missing data in the text modality, and also cases where the text is rarely informative. This is a common case in real-world tabular-text datasets, and it is important to have it represented in our benchmark.
- We reverse the ordinal encoding of the target.
- We reverse the name change of "revenue" back to "annual_inc".
- We also create a sec_app_fico_n like the fico_n from anz-Guerrero et al. (2025).
- We drop rows where "application_type" is missing as these rows have consistent missing values across features and likely represent some data loading artifact.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Default",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Default",
    time_on="issue_d",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

original_df = pd.read_csv(dataset_mold.path /  "accepted_2007_to_2018Q4.csv.gz")
df = pd.read_csv(dataset_mold.path / "LC_loans_granting_model_dataset.csv?download=1")
print("Loaded data shape:", df.shape)
print("Original data shape:", original_df.shape)

# 0 for fully paid loans and a 1 for defaulted loans.
df["Default"] = df["Default"].map({0: "Fully Paid", 1: "Defaulted"})

df = df.rename(columns={"revenue": "annual_inc"})

# Transform     "sec_app_fico_range_low"    "sec_app_fico_range_high", to sec_app_fico_n
original_df["sec_app_fico_n"] = (original_df["sec_app_fico_range_low"] + original_df["sec_app_fico_range_high"]) / 2

new_features_from_original = original_df[[
    "id",
    # known when the loan is granted, so it can be used as a feature without data leakage.
    #   This, however, would create a different task, in that we want to train a model that the company uses after their own risk assessment but before granting the loan.
    #   Given that these values and grades are often determined by an internal model of the company, they might also introduce an unknown bias.
    #   We stick to treating the task as a pre-assessment modelling. An alternative version could also function as a "stacking" model that takes the output of the company's internal model as an input.
    #   The variables for this task are:
    # "term",
    # "int_rate",
    # "installment",
    # "grade",
    # "sub_grade",
    # "verification_status",
    # Known at the start / from data source even without the company assessing the risk, so it can be used no matter the use case.
    "emp_title",
    "disbursement_method",
    "acc_now_delinq",
    "acc_open_past_24mths",
    "all_util",
    "sec_app_fico_n",
    # annual_inc	annual_inc_joint -> were already merged by prior work
    # fico_range_high fico_range_low -> transformed before into avg fico
    "application_type", "avg_cur_bal", "bc_open_to_buy", "bc_util", "chargeoff_within_12_mths", "collections_12_mths_ex_med", "delinq_2yrs", "delinq_amnt", "earliest_cr_line", "il_util", "inq_fi", "inq_last_12m", "inq_last_6mths", "max_bal_bc", "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl", "mort_acc", "mths_since_last_delinq", "mths_since_last_major_derog", "mths_since_last_record", "mths_since_rcnt_il", "mths_since_recent_bc", "mths_since_recent_bc_dlq", "mths_since_recent_inq", "mths_since_recent_revol_delinq", "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl", "num_bc_sats", "num_bc_tl", "num_il_tl", "num_op_rev_tl", "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats", "num_tl_120dpd_2m", "num_tl_30dpd", "num_tl_90g_dpd_24m", "num_tl_op_past_12m", "open_acc", "open_acc_6m", "open_il_12m", "open_il_24m", "open_act_il", "open_rv_12m", "open_rv_24m", "pct_tl_nvr_dlq", "percent_bc_gt_75", "pub_rec", "pub_rec_bankruptcies", "revol_bal", "revol_util", "tax_liens", "tot_coll_amt", "tot_cur_bal", "tot_hi_cred_lim", "total_acc", "total_bal_ex_mort", "total_bal_il", "total_bc_limit", "total_cu_tl", "total_il_high_credit_limit", "total_rev_hi_lim", "revol_bal_joint", "sec_app_earliest_cr_line", "sec_app_inq_last_6mths", "sec_app_mort_acc", "sec_app_open_acc", "sec_app_revol_util", "sec_app_open_act_il", "sec_app_num_rev_accts", "sec_app_chargeoff_within_12_mths", "sec_app_collections_12_mths_ex_med", "sec_app_mths_since_last_major_derog", "sec_app_fico_range_low", "sec_app_fico_range_high", # dti dti_joint -> already merged by prior work via max
]]

# Merge new features into the curated version of the dataset.
df = df.merge(new_features_from_original, on="id", how="left")
del new_features_from_original, original_df

as_cat_type = [
    # There are only two options on this website. This is ordinal technically, so we could treat it as a number too.
    "home_ownership_n",
    "application_type",
    "disbursement_method",
    "emp_length",
    "Default",
    "experience_c",
    "purpose",
]
as_string_type = [
    "emp_title",
    "addr_state",
    "zip_code",
    "title",
    "desc",
]

df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["sec_app_earliest_cr_line"] = pd.to_datetime(df["sec_app_earliest_cr_line"], format="%b-%Y")
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y")

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")


df[as_cat_type] = df[as_cat_type].astype("category")

df = df[~df["application_type"].isna()]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.drop(columns=[
    "id",  # meaningless identifier
    "experience_c", # constant after preprocessing
])

/tmp/ipykernel_613182/1013934924.py:4: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  original_df = pd.read_csv(dataset_mold.path /  "accepted_2007_to_2018Q4.csv.gz")


/tmp/ipykernel_613182/1013934924.py:5: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_mold.path / "LC_loans_granting_model_dataset.csv?download=1")


Loaded data shape: (1347681, 15)
Original data shape: (2260701, 151)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
    duplicate_column_check=False, # skip
)


#### Dataset Overview
Rows: 1,310,259
Columns: 97
Use sampling: True (sample size: 131,026)
Get missing and unique counts per column...


missing/unique per-col:   0%|          | 0/97 [00:00<?, ?it/s]

Get example values per column...


examples per-col:   0%|          | 0/97 [00:00<?, ?it/s]

Get numeric feature statistics...


numeric stats:   0%|          | 0/83 [00:00<?, ?it/s]

Get cat stats...


cat stats:   0%|          | 0/14 [00:00<?, ?it/s]

Get target stats...
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['tot_hi_cred_lim', 'tot_cur_bal', 'emp_title', 'total_bal_ex_mort', 'total_il_high_credit_limit', 'total_bal_il', 'desc', 'revol_bal', 'avg_cur_bal', 'bc_open_to_buy']


Rows remaining as candidates after top-10 filter: 203 (of 1,310,259)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,issue_d,annual_inc,dti_n,loan_amnt,fico_n,emp_length,purpose,home_ownership_n,addr_state,zip_code,Default,title,desc,emp_title,disbursement_method,acc_now_delinq,acc_open_past_24mths,all_util,sec_app_fico_n,application_type,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,collections_12_mths_ex_med,delinq_2yrs,delinq_amnt,earliest_cr_line,il_util,inq_fi,inq_last_12m,inq_last_6mths,max_bal_bc,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_last_delinq,mths_since_last_major_derog,mths_since_last_record,mths_since_rcnt_il,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,open_acc,open_acc_6m,open_il_12m,open_il_24m,open_act_il,open_rv_12m,open_rv_24m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec,pub_rec_bankruptcies,revol_bal,revol_util,tax_liens,tot_coll_amt,tot_cur_bal,tot_hi_cred_lim,total_acc,total_bal_ex_mort,total_bal_il,total_bc_limit,total_cu_tl,total_il_high_credit_limit,total_rev_hi_lim,revol_bal_joint,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,sec_app_fico_range_low,sec_app_fico_range_high
0,2014-04-01,61000.0,10.06,25850,682.0,10+ years,debt_consolidation,RENT,GA,300xx,Fully Paid,Debt consolidation,<NA>,Asset Manager,Cash,0.0,1.0,NaN,NaN,Individual,1259.0,7882.0,65.3,0.0,0.0,0.0,0.0,1989-05-01,NaN,NaN,NaN,0.0,NaN,NaN,299.0,17.0,17.0,0.0,NaN,NaN,52.0,NaN,67.0,NaN,12.0,NaN,0.0,8.0,11.0,8.0,8.0,0.0,13.0,13.0,11.0,13.0,0.0,0.0,0.0,0.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,100.0,37.5,1.0,0.0,16365.0,36.0,1.0,0.0,16365.0,45400.0,13.0,16365.0,NaN,22700.0,NaN,0.0,45400.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-05-01,115000.0,8.31,28000,692.0,10+ years,debt_consolidation,RENT,NY,112xx,Fully Paid,Debt consolidation,<NA>,System Administrator,Cash,0.0,2.0,NaN,NaN,Individual,49682.0,8123.0,82.1,0.0,0.0,0.0,0.0,2003-09-01,NaN,NaN,NaN,0.0,NaN,110.0,140.0,4.0,4.0,2.0,NaN,NaN,NaN,NaN,24.0,NaN,23.0,NaN,0.0,5.0,6.0,5.0,7.0,1.0,6.0,10.0,6.0,7.0,0.0,0.0,0.0,1.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,100.0,80.0,0.0,0.0,37318.0,80.6,0.0,0.0,347773.0,376300.0,13.0,37318.0,NaN,45300.0,NaN,0.0,46300.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-02-01,30000.0,8.68,12000,702.0,NI,credit_card,OWN,CA,952xx,Fully Paid,Credit card refinancing,credit card consolidation.,<NA>,Cash,0.0,3.0,NaN,NaN,Individual,1167.0,14004.0,44.0,0.0,0.0,0.0,0.0,1963-10-01,NaN,NaN,NaN,0.0,NaN,127.0,604.0,8.0,8.0,0.0,NaN,NaN,67.0,NaN,9.0,NaN,NaN,NaN,0.0,5.0,7.0,8.0,9.0,2.0,10.0,11.0,7.0,10.0,0.0,0.0,0.0,2.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,100.0,12.5,4.0,0.0,11672.0,40.4,4.0,0.0,11672.0,28900.0,13.0,11672.0,NaN,25000.0,NaN,0.0,28900.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-12-01,85000.0,7.58,3000,667.0,< 1 year,home_improvement,MORTGAGE,SC,294xx,Fully Paid,Home improvement,<NA>,Project Manager,Cash,0.0,5.0,33.0,NaN,Individual,43592.0,1796.0,62.6,0.0,0.0,0.0,0.0,2006-04-01,35.0,0.0,5.0,0.0,2304.0,116.0,115.0,8.0,3.0,4.0,65.0,66.0,39.0,49.0,8.0,NaN,3.0,NaN,1.0,2.0,3.0,2.0,4.0,2.0,5.0,7.0,3.0,7.0,0.0,0.0,0.0,2.0,7.0,1.0,0.0,0.0,1.0,1.0,4.0,92.3,0.0,1.0,0.0,3274.0,28.7,0.0,726.0,305144.0,327236.0,13.0,10644.0,7370.0,4800.0,1.0,21336.0,11400.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-11-01,81435.0,17.27,25200,677.0,10+ years,debt_consolidation,MORTGAGE,VA,224xx,Defaulted,Debt consolidation,<NA>,Assistant Principal,Cash,0.0,6.0,NaN,NaN,Individual,39249.0,1392.0,88.7,0.0,0.0,0.0,0.0,2000-09-01,NaN,NaN,NaN,0.0,NaN,148.0,170.0,8.0,8.0,5.0,NaN,NaN,NaN,NaN,75.0,NaN,7.0,NaN,0.0,4.0,8.0,4.0,8.0,12.0,8.0,13.0,8.0,11.0,0.0,0.0,0.0,

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,emp_length,category,0.0,0.00,12.0,"10+ years, 2 years, 3 years, < 1 year, 1 year, 5 years, 4 years, NI, 6 years, 8 years"
1,purpose,category,0.0,0.00,14.0,"debt_consolidation, credit_card, home_improvement, other, major_purchase, medical, small_business, car, moving, vacation"
2,home_ownership_n,category,0.0,0.00,4.0,"MORTGAGE, RENT, OWN, OTHER"
3,Default,category,0.0,0.00,2.0,"Fully Paid, Defaulted"
4,disbursement_method,category,0.0,0.00,2.0,"Cash, DirectPay"
5,application_type,category,0.0,0.00,2.0,"Individual, Joint App"
6,sec_app_earliest_cr_line,datetime64[ns],1292511.0,98.65,570.0,"2006-03-01 00:00:00, 2006-08-01 00:00:00, 2004-07-01 00:00:00, 2005-06-01 00:00:00, 2004-12-01 00:00:00, 2007-08-01 00:00:00, 2004-08-01 00:00:00, 2002-08-01 00:00:00, 2005-10-01 00:00:00, 2005-12-01 00:00:00"
7,issue_d,datetime64[ns],0.0,0.00,129.0,"2016-03-01 00:00:00, 2015-10-01 00:00:00, 2015-07-01 00:00:00, 2015-12-01 00:00:00, 2014-10-01 00:00:00, 2016-02-01 00:00:00, 2015-11-01 00:00:00, 2015-04-01 00:00:00, 2015-08-01 00:00:00, 2015-01-01 00:00:00"
8,earliest_cr_line,datetime64[ns],0.0,0.00,739.0,"2002-08-01 00:00:00, 2001-08-01 00:00:00, 2001-10-01 00:00:00, 2002-09-01 00:00:00, 2003-09-01 00:00:00, 2000-08-01 00:00:00, 2002-10-01 00:00:00, 2000-10-01 00:00:00, 2000-09-01 00:00:00, 2004-09-01 00:00:00"
9,sec_app_mths_since_last_major_derog,float64,1303942.0,99.52,112.0,"1.0, 4.0, 44.0, 9.0, 33.0, 2.0, 12.0, 60.0, 55.0, 51.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
annual_inc,131026.0,77432.859017,66269.682201,2000.0,7000000.0
dti_n,131026.0,18.351067,10.941811,0.0,999.0
loan_amnt,131026.0,14494.346160,8723.635823,1000.0,40000.0
fico_n,131026.0,698.092077,31.828085,662.0,847.5
acc_now_delinq,131026.0,0.005136,0.078604,0.0,6.0
acc_open_past_24mths,126783.0,4.696111,3.187263,0.0,50.0
all_util,51366.0,58.046704,20.950480,0.0,184.0
sec_app_fico_n,1781.0,667.870298,47.222585,542.0,832.0
avg_cur_bal,124690.0,13487.800160,16240.565864,0.0,646339.0
bc_open_to_buy,125438.0,10206.234769,15444.151180,0.0,320558.0


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column                   rank                                        
Default                  1                  Fully Paid  104830  80.01
                         2                   Defaulted   26196  19.99
addr_state               1                          CA   18981  14.49
                         2                          TX   10841   8.27
                         3                          NY   10729   8.19
                         4                          FL    9316   7.11
                         5                          IL    4919   3.75
application_type         1                  Individual  128572  98.13
                         2                   Joint App    2454   1.87
desc                     1                        <NA>  119191  90.97
                         2         Debt consolidation.     117   0.09
                         3         Debt Consolidation.      76   0.06
                         4         debt consolidation.      55   0.04
                         5       Pay off credit cards.      28   0.02
disbursement_method      1                        Cash  130382  99.51
                         2                   DirectPay     644   0.49
earliest_cr_line         1         2002-08-01 00:00:00     911    0.7
                         2         2001-08-01 00:00:00     902   0.69
                         3         2001-10-01 00:00:00     885   0.68
                         4         2002-09-01 00:00:00     882   0.67
                         5         2003-09-01 00:00:00     875   0.67
emp_length               1                   10+ years   43186  32.96
                         2                     2 years   11747   8.97
                         3                     3 years   10616    8.1
                         4                    < 1 year   10498   8.01
                         5                      1 year    8470   6.46
emp_title                1                        <NA>    8393   6.41
                         2                     Teacher    2116   1.61
                         3                     Manager    1908   1.46
                         4                       Owner     976   0.74
                         5            Registered Nurse     848   0.65
home_ownership_n         1                    MORTGAGE   65022  49.63
                         2                        RENT   52035  39.71
                         3                         OWN   13921  10.62
                         4                       OTHER      48   0.04
issue_d                  1         2016-03-01 00:00:00    4594   3.51
                         2         2015-10-01 00:00:00    4328    3.3
                         3         2015-07-01 00:00:00    4156   3.17
                         4         2015-12-01 00:00:00    3883   2.96
                         5         2014-10-01 00:00:00    3685   2.81
purpose                  1          debt_consolidation   76137  58.11
                         2                 credit_card   28842  22.01
                         3            home_improvement    8559   6.53
                         4                       other    7409   5.65
                         5              major_purchase    2854   2.18
sec_app_earliest_cr_line 1                        <NA>  129245  98.64
                         2         2006-03-01 00:00:00      21   0.02
                         3         2006-08-01 00:00:00      19   0.01
                         4         2004-07-01 00:00:00      19   0.01
                         5         2005-06-01 00:00:00      18   0.01
title                    1          Debt consolidation   64840  49.49
                         2     Credit card refinancing   24527  18.72
                         3            Home improvement    7343    5.6
                         4                       Other    6337   4.84
                         5              Major purchase    2319   1.77
zip_code                 1                       945xx    1

In [8]:
# Target Distribution
target_df

,count,pct
Default,,
Fully Paid,1049280,80.08
Defaulted,260979,19.92


## Task Curation

In [9]:
time_df = df.copy()
time_df["year_month"] = time_df[task_mold.time_on].dt.to_period("Y")

# 1) Total number of samples per month
monthly_totals = (
    time_df
    .groupby("year_month")
    .size()
    .rename("total_samples")
)

# Optional: sort by month and convert PeriodIndex to timestamp (month start)
result = monthly_totals.sort_index()
result.index = result.index.to_timestamp()
result

year_month
2008-01-01       836
2009-01-01      4716
2010-01-01     11536
2011-01-01     19776
2012-01-01     51396
2013-01-01    132915
2014-01-01    221034
2015-01-01    372542
2016-01-01    276431
2017-01-01    164085
2018-01-01     54992
Freq: YS-JAN, Name: total_samples, dtype: int64

In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import subsample_temporal
# Sort by time
df = df.sort_values(by=task_mold.time_on).reset_index(drop=True)


ref_date  = pd.Timestamp("2016")
train_index = df[
    df[task_mold.time_on] < ref_date
].index.tolist()
test_index = df[
    (df[task_mold.time_on].dt.year == ref_date.year)
].index.tolist()

df, test_index, test_index = subsample_temporal(
    df=df,
    train_idx=train_index,
    test_idx=test_index,
    stratify_on=task_mold.stratify_on,
)

splits = {0: {
    0: (train_index, test_index)
}}

print(f"Split: Train size: {len(train_index)}, Test size: {len(test_index)}")
assert df[task_mold.time_on].iloc[train_index].max() < df[task_mold.time_on].iloc[test_index].min()

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="""We try to create splits that simulate a model deployed to solve the task.

The official data is updated monthly but has not enough data per month to create large enough test splits. We opt for simulating a model that is refit every year to obtain a robust test set instead. This introduces the unrealistic downside of data shift across a year that would not exist in a real-world model. We create 1 test split (2016). We use all data before the test year as training data.
""",
    splits=splits,
    time_horizon=1,
    time_horizon_unit="years",
)

Split: Train size: 814751, Test size: 250000


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to lending_club/versions/019d738c-5edd-7437-9116-e3bc87a8b0c5


019d738c-5edd-7437-9116-e3bc87a8b0c5
c80c2cea9dfb4b67e6ba42ff9d4c96fcd93aeba045bfc33904ccbbba608c6d52
